# cell-eval Benchmark: Standardized Perturbation Prediction Metrics

Runs ArcInstitute's `cell-eval` metrics on BioJEPA's perturbation predictions for
direct comparison against X-Cell, STATE, scGPT, Cell2Sentence.

**Output**: `~/data/jepa/v0_7/cell_eval_results/`

In [1]:
import sys
import json
import gc
import torch
import random
import numpy as np
import pandas as pd
import anndata as ad
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd()))

from biojepa_v0_7 import BioJepa, BioJepaConfig
from training_v0_7 import create_model, maybe_compile, load_feature_banks
from config_v0_7 import VERSION, DataConfig
from evals.evals import get_seq_embeddings, get_target_embeddings
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig
from cell_eval_utils import load_pert_name_mappings, build_cell_eval_adata

## Config

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/jepa/v0_7').expanduser()
ref_root = Path('~/data/jepa/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'eval_results'
)

shard_dir = data_root / 'predictor_t'
output_dir = data_root / 'cell_eval_results'
output_dir.mkdir(parents=True, exist_ok=True)

N_NTC = 64
MAX_SAMPLES_PER_DATASET = 50000
BATCH_SIZE = 64
DATASETS = ['adamson', 'k562e_raw', 'norman', 'rep1e', 'k562gw', 'sciplex']

using cpu


## Load Model

In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=2, #6
    heads=2, #4
    embed_dim=8, #256
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=2.38,
    film_linear_multiple=0.81,
    sim_coeff=25,
    std_coeff=25,
    cov_coeff=1,
    pert_latent_dim=128,
    pert_mode_dim=64,
)

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

Student/Teacher: 84,086
ACpredictor: 85,824
PerturbationComposer: 510,400


### Load Model

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_ac_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Load Decoder

In [6]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

<All keys matched successfully>

### Load Feature Banks

In [7]:
seq_banks, target_bank = load_feature_banks(data_cfg, device)

with open(data_root / 'gene_names.json') as f:
    gene_names = json.load(f)

len(gene_names)

Loaded DNA embeddings: torch.Size([11643, 1536])
Loaded chemical embeddings: torch.Size([188, 1536])
Loaded target embeddings: torch.Size([9975, 320])


10000

## Perturbation Name Mappings

In [8]:
pert_dir = data_root / 'pert_embd'
dna_map, chem_map, target_map = load_pert_name_mappings(pert_dir)
print(f'DNA mappings: {len(dna_map)}, Chemical: {len(chem_map)}, Target: {len(target_map)}')

DNA mappings: 11643, Chemical: 188, Target: 9975


## Run cell-eval Per Dataset

In [ ]:
from cell_eval import MetricsEvaluator

all_agg = {}
skipped = {}

for ds in DATASETS:
    print(f'\n{"=" * 60}')
    print(f'{ds}')
    print(f'{"=" * 60}')

    ntc_path = data_root / 'ntc_controls' / f'{ds}_ntc.npz'
    if not ntc_path.exists():
        print(f'  SKIP: NTC file not found at {ntc_path}')
        skipped[ds] = 'NTC file not found'
        continue

    shards = sorted((shard_dir / 'test').glob(f'shard_{ds}_test_*.npz'))
    if not shards:
        print(f'  SKIP: No test shards for {ds}')
        skipped[ds] = 'no test shards'
        continue

    adata_pred, adata_real = build_cell_eval_adata(
        ds, model, decoder, seq_banks, target_bank, gene_names,
        data_root, shard_dir, dna_map, chem_map, target_map,
        get_seq_embeddings, get_target_embeddings,
        n_ntc=N_NTC, max_samples=MAX_SAMPLES_PER_DATASET,
        device=str(device), batch_size=BATCH_SIZE,
    )

    n_perts = adata_pred.obs['perturbation'].nunique() - 1
    n_pred = (adata_pred.obs['perturbation'] != 'control').sum()
    n_real = (adata_real.obs['perturbation'] != 'control').sum()
    print(f'  {n_perts} perturbations, {n_pred} predicted cells, {n_real} real cells')

    ds_dir = output_dir / ds
    ds_dir.mkdir(parents=True, exist_ok=True)
    adata_pred.write_h5ad(ds_dir / 'pred.h5ad')
    adata_real.write_h5ad(ds_dir / 'real.h5ad')
    print(f'  Saved AnnData to {ds_dir}')

    try:
        evaluator = MetricsEvaluator(
            adata_pred=adata_pred,
            adata_real=adata_real,
            control_pert='control',
            pert_col='perturbation',
        )
        results, agg_results = evaluator.compute()

        results.write_csv(str(ds_dir / 'per_pert_results.csv'))
        agg_results.write_csv(str(ds_dir / 'agg_results.csv'))
        all_agg[ds] = agg_results
        print(f'  Results saved to {ds_dir}')
    except (ValueError, Exception) as e:
        reason = f'MetricsEvaluator failed: {e}'
        print(f'  SKIP eval: {reason}')
        skipped[ds] = reason

    del adata_pred, adata_real
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if skipped:
    print(f'\nSkipped datasets: {skipped}')


adamson
adamson: 11 perturbations, 4288 real samples, 64 NTC controls


adamson perts: 100%|███████████████████████████████████████████████| 11/11 [00:14<00:00,  1.35s/it]


adamson: collecting real expression (streaming, max_samples=50000)...
adamson: adata_pred=768 rows, adata_real=4352 rows, 9749/10000 genes measured


... storing 'perturbation' as categorical
... storing 'perturbation' as categorical


  11 perturbations, 704 predicted cells, 4288 real cells


...

/opt/miniconda3/envs/general/lib/python3.13/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/miniconda3/envs/general/lib/python3.13/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
... storing 'perturbation' as categorical


  986 perturbations, 67392 predicted cells, 49471 real cells


... storing 'perturbation' as categorical


  Saved AnnData to /Users/djemec/data/jepa/v0_7/cell_eval_results/k562gw


INFO:cell_eval.utils:Data appears to be log1p normalized (decimals detected, range [0.00, 7.64])
INFO:cell_eval._evaluator:Input is found to be log-normalized already - skipping transformation.
INFO:cell_eval.utils:Data appears to be log1p normalized (decimals detected, range [0.00, 6.33])
INFO:cell_eval._evaluator:Input is found to be log-normalized already - skipping transformation.
INFO:cell_eval._evaluator:Computing DE for real data
INFO:cell_eval._evaluator:Using the following pdex kwargs: {'reference': 'control', 'groupby_key': 'perturbation', 'num_workers': 10, 'batch_size': 100, 'metric': 'wilcoxon', 'is_log1p': True, 'as_polars': True}
INFO:pdex._single_cell:Log1p status: True
INFO:pdex._single_cell:Precomputing masks for each target gene
Identifying target masks: 100%|███████████████████████████████| 987/987 [00:00<00:00, 56525.77it/s]
INFO:pdex._single_cell:Precomputing variable indices for each feature
Identifying variable indices: 100%|███████████████████████| 8248/8248 [0

## Results Summary

In [ ]:
import polars as pl

summary_rows = []
for ds in DATASETS:
    csv_path = output_dir / ds / 'agg_results.csv'
    if not csv_path.exists():
        continue
    agg = pl.read_csv(csv_path)
    row = {'dataset': ds}
    for r in agg.iter_rows(named=True):
        if 'metric' in r and 'mean' in r:
            row[r['metric']] = r['mean']
    summary_rows.append(row)

pd.DataFrame(summary_rows).set_index('dataset')

## Save Combined Report

In [ ]:
combined = {}
for ds in DATASETS:
    csv_path = output_dir / ds / 'agg_results.csv'
    if not csv_path.exists():
        continue
    agg = pl.read_csv(csv_path)
    ds_metrics = {}
    for r in agg.iter_rows(named=True):
        if 'metric' in r and 'mean' in r:
            ds_metrics[r['metric']] = float(r['mean'])
    combined[ds] = ds_metrics

report = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'model_version': VERSION,
    'checkpoint': str(checkpoint_path),
    'n_ntc': N_NTC,
    'max_samples_per_dataset': MAX_SAMPLES_PER_DATASET,
    'by_dataset': combined,
}
if skipped:
    report['skipped'] = skipped

report_path = output_dir / 'cell_eval_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

report_path

## Score Against Mean-Prediction Baseline

Generates a mean-prediction baseline (per-perturbation mean of real data) for each dataset,
runs cell-eval on it, then normalizes BioJEPA's results against that baseline using
`score_agg_metrics`. Scores range from 0 (equal to baseline) to 1 (perfect).

In [ ]:
from cell_eval import build_base_mean_adata, score_agg_metrics

baseline_skipped = {}
all_scores = {}

for ds in DATASETS:
    ds_dir = output_dir / ds
    model_agg_path = ds_dir / 'agg_results.csv'
    real_path = ds_dir / 'real.h5ad'

    if not model_agg_path.exists() or not real_path.exists():
        baseline_skipped[ds] = 'no model results or real.h5ad'
        continue

    print(f'\n{ds}: building mean-prediction baseline...')
    adata_real = ad.read_h5ad(real_path)

    try:
        baseline_pred = build_base_mean_adata(
            adata_real, pert_col='perturbation', control_pert='control',
        )

        baseline_evaluator = MetricsEvaluator(
            adata_pred=baseline_pred,
            adata_real=adata_real,
            control_pert='control',
            pert_col='perturbation',
        )
        _, baseline_agg = baseline_evaluator.compute(write_csv=False)
        baseline_agg.write_csv(str(ds_dir / 'baseline_agg_results.csv'))

        scores = score_agg_metrics(
            results_user=pl.read_csv(str(model_agg_path)),
            results_base=baseline_agg,
        )
        scores.write_csv(str(ds_dir / 'scored_vs_baseline.csv'))
        all_scores[ds] = scores
        print(f'  avg_score: {scores.filter(pl.col("metric") == "avg_score")["from_baseline"][0]:.4f}')
    except (ValueError, Exception) as e:
        reason = f'baseline scoring failed: {e}'
        print(f'  SKIP: {reason}')
        baseline_skipped[ds] = reason

    del adata_real
    gc.collect()

if baseline_skipped:
    print(f'\nBaseline scoring skipped: {baseline_skipped}')

In [ ]:
score_rows = []
for ds, scores in all_scores.items():
    row = {'dataset': ds}
    for r in scores.iter_rows(named=True):
        row[r['metric']] = round(r['from_baseline'], 4)
    score_rows.append(row)

pd.DataFrame(score_rows).set_index('dataset')

In [ ]:
del model, decoder, seq_banks, target_bank
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()